# 03 系列モデル（GRU・1D-CNN）— 生の時系列から、隠れた状態を読めるか

> 🔰 **やさしい導入**（この回のひとこと）
>
> - **ひとことで言うと**: 過去30日の並びを、そのまま読むネット（GRU）で、見えない状態（よごれ）を読めるかを試す回です。
> - **たとえ話**: 毎日の日記を順に読み、記憶を更新していく。「しばらく雨が降ってないからパネルが汚れてそうだ」と気づく。
> - **先に読むと分かりやすい**: [00_basics.ipynb](00_basics.ipynb) の「2-1・3（用語集：隠れた状態・GRU）」。用語が分からなくなったら、同じNotebookの「用語集」へ。
> - **難しければ**: 各コードの後ろにある「**出力の見方**」の文章だけ読めば、結論は追えます。

> 🔒 すべて**合成データ**です。⏱ 所要時間: 説明 15分 ＋ 実行 約1分（比較表は事前に `src/run_phase3.py`（約15分）を実行した結果のCSVを読みます）

## ゴール
- **GRU**（系列を順に読む）と **1D-CNN**（窓を畳み込む）に、**手作りの特徴量なし**で、生の窓（過去30日）を渡して学習させる。
- 隠れた状態（**よごれ**）を、モデルが**自力で読める**かを確かめる。
- 予報が悪い普通の世界と、予報が完璧な診断用の世界で、**木・MLP と比べる**。
- 隠れた状態が**無い**対照の世界で、系列モデルの「見かけの強さ」が消えることを確かめる。

## 前のNotebookとのつながり
01 で、手作りの特徴量（性能比・雨の降らない日数など）を使えば、予報が完璧な世界では隠れた状態が読めることが分かりました（HGB 0.284→0.218）。02 で、MLP は木と同等でした。
では、**特徴量を手で作らず**、生の窓から**自分で見つけてくれる**のが、系列モデルの売りです。本当にそうでしょうか。

## 入力
- **窓**: 起点日（対象日のL日前）までの W 日。1日あたり9チャンネル（天気one-hot・日照率・日射・気温・発電量・季節sin/cos）。
- **静的入力**: 対象日の予報 fcL、対象日の季節。

## 3つの世界
| 世界 | 予報 | 隠れた状態（よごれ・温度） |
|---|---|---|
| A 普通 | 通常（不正確） | あり |
| B 予報が完璧 | 真の日射量 | あり |
| C 予報が完璧・隠れた状態なし | 真の日射量 | **なし**（対照） |

## キーワード
- **GRU**: 系列を1日ずつ読み、内部の「記憶」を更新していくネット。よごれのように、日々積み重なる状態に向く。
- **1D-CNN**: 窓の中の短いパターンを、フィルタで拾うネット。
- **窓MLP**: 窓を1列に並べただけのMLP。時間の並びの構造を使わない。
- **対照実験**: 伸びしろが無い世界で試して、「強さ」が本物か確かめる。

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import run_phase3
from tsf_eval import errors, linear_fc
from tsf_models import build_windows, seq_fit_predict

plt.rcParams["font.family"] = ["Hiragino Sans", "AppleGothic", "DejaVu Sans"]
pd.set_option("display.width", 220)

tr, va = run_phase3._world(1_000_001, "B 予報が完璧")
seq, stat, valid = build_windows(va, 1, 30)
print("窓の形（日数, 窓の長さ, チャンネル）:", seq.shape, "／ 静的入力:", stat.shape)

**出力の見方**: 検証期間の730日それぞれについて、過去30日 × 9チャンネルの窓と、3個の静的入力（予報・季節sin/cos）を作りました。**窓は起点日（対象日の1日前）までしか含みません**（テストで確認済み）。

In [ ]:
p_gru = seq_fit_predict("gru", tr, va, 1, seed=0, n_seeds=3)
p_lin = linear_fc(tr, va, 1)
keep = ~np.isnan(p_gru) & (va["gen_kwh"].to_numpy() > 2.0)
rel = lambda p: (p[keep] - va["gen_kwh"].to_numpy()[keep]) / va["gen_kwh"].to_numpy()[keep]
fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for a, p, name in zip(ax, [p_lin, p_gru], ["予報の直線回帰", "GRU（生の窓）"]):
    a.scatter(100 * va["soiling"].to_numpy()[keep], 100 * rel(p), s=6, alpha=0.5)
    a.axhline(0, color="k", lw=0.6)
    a.set_title(f"{name}  MAE={errors(p, va['gen_kwh'])['MAE']:.3f}  相関={np.corrcoef(rel(p), va['soiling'].to_numpy()[keep])[0, 1]:.2f}")
    a.set_xlabel("真のよごれ（％）。モデルには見えない"); a.grid(alpha=0.3)
ax[0].set_ylabel("予測の割合の誤差（％）。正=高めに予測")
plt.tight_layout(); plt.show()

**出力の見方**（予報が完璧な世界B・1つの世界）:
- **直線回帰**（左）は、よごれが多いほど**高めに外れます**（右上がりの傾向）。直線回帰は、よごれを知らないので、**よごれの分だけ予測が高すぎる**。
- **GRU**（右）は、よごれの量によらず、誤差がほぼ**水平に散らばっています**。**よごれを読んで補正している**ことになります。
- GRUには、よごれの値は**与えていません**。過去30日の天気・日射・発電量の並びから、**自分で見つけた**のです。

In [ ]:
OUT = ROOT / "outputs"
need = [OUT / f"phase3_{k}.csv" for k in ("E1", "E2", "E3")]
if not all(p.exists() for p in need):
    print("先に .venv/bin/python src/run_phase3.py を実行してください（約15分）")
else:
    e1, e2, e3 = (pd.read_csv(p) for p in need)
    display(e1.pivot(index="窓の長さW（日）", columns="世界", values="GRUのMAE"))

**出力の見方**（窓の長さ W と GRU のMAE。3つの世界の平均）:
- 世界A（普通）: W=7 で 0.793、W=14 で 0.772、W=30 で 0.776、W=60 で 0.771。**14日以上では、ほぼ変わりません**（標準誤差 0.007〜0.017）。
- 世界B（予報が完璧）: W=7 で 0.234、14 で 0.228、30 で 0.218、60 で 0.213。**窓が長いほど少し良くなります**。よごれは、**雨が降らない日数が長く続くほど溜まる**ので、長い窓で読めるためです。
- 窓7日では足りず、**30日あれば十分に近い**。以降は W=30 を使います。

In [ ]:
if all(p.exists() for p in need):
    piv = e2.pivot(index="手法", columns="世界", values="MAE")
    display(piv.loc[["予報の直線回帰", "HGB（手作り特徴量・全部）", "HGB（予報・季節だけ）", "MLP（予報・季節だけ）",
                     "窓MLP（生の窓を並べただけ）", "GRU（生の窓）", "1D-CNN（生の窓）", "上限2（隠れた状態まで知っている）"]])

**出力の見方**（L=1・5つの世界・各手法は3つの初期値の平均。MAE）:

**世界A（普通）**:
- **GRU 0.774**、**MLP（予報・季節だけ）0.770**、HGB（全部）0.790、HGB（予報・季節だけ）0.798、1D-CNN 0.802、直線回帰 0.865。
- **GRUとMLP（予報・季節だけ）は区別できません**。HGB（全部）に対して、GRUは −0.016（標準誤差0.005）、MLP（予報・季節だけ）は −0.020（0.008）。
- **窓MLP（0.921）は大きく悪化**。窓を1列に並べただけでは、パラメータが多すぎて過学習する（直線回帰よりも悪い）。**時間の構造を使うGRUと、使わない窓MLPで、差は 0.15**。
- 予報が不正確な世界では、**隠れた状態を読んでも、予報の誤差に埋もれます**。系列モデルの利点は、ほとんど出ません。

**世界B（予報が完璧）**:
- **GRU 0.210**が最良。HGB（手作り特徴量・全部）0.214 と並びます（差 −0.005、標準誤差 0.002。約2.5倍ですが、差は小さい）。**手作りの特徴量と同じ水準を、生の窓から自力で達成**しました。
- 上限2は 0.190。GRUは、**その 0.02 上**まで来ています。
- 予報・季節だけを使う HGB（0.274）・MLP（0.267）は、それより **0.06 悪い**。**過去の並びを読むことの価値が、ここで初めて明確に出ます**。
- 1D-CNN は 0.234、窓MLPは 0.321。**GRUのほうが、積み重なる状態に向いている**。

**世界C（予報が完璧・隠れた状態なし）**: 対照実験。
- すべての手法が **0.205〜0.218** に収まり、**GRU 0.218 は、直線回帰（0.205）・MLP（0.209）より悪い**。隠れた状態が無いと、**系列モデルの優位は消えます**（むしろ、不要な複雑さのぶん少し悪い）。
- 窓MLPだけは 0.323 と悪いまま。
- つまり、世界BでのGRUの強さは、「過去の窓を読んだから」であって、「ただ複雑なモデルだから」ではない。

In [ ]:
if all(p.exists() for p in need):
    display(e3)

**出力の見方**（世界B・5つの世界の平均）:
- **予測の割合の誤差**と、**真のよごれ**の相関: 直線回帰 **0.42**、HGB（予報・季節だけ）**0.49**、**GRU 0.07**。直線回帰・HGBは、よごれの分だけ系統的に高く外れる。**GRUは、ほぼ無相関**で、よごれを補正している。
- 温度の低下との相関は、いずれも小さい（0.01〜0.04）。温度の低下は、予報と季節から（ある程度）推測できるため。
- 割合の誤差の平均は、GRU **0.026**、HGB 0.033、直線回帰 0.035。

## 試してみよう（演習）
1. **窓を短く**: `seq_fit_predict("gru", ..., window=3)` にして、世界BのMAEを見てください。分かるのは、短すぎる窓ではよごれが読めないことです。
2. **隠れ層を変える**: `hidden=8` と `hidden=64` で比べてください。分かるのは、GRUの大きさと過学習の関係です。
3. **よごれを速く溜める**: `make_series(..., soiling_rate=0.015)` の世界Bで、窓MLPとGRUの差がどう変わるか見てください。
4. **データを減らす**: `split_series(df, train_years=2)` で学習期間を2年にして、GRUと窓MLPのMAEを比べてください（次の Phase4 の予告です）。

## まとめ
- GRUは、**手作りの特徴量なしで、隠れた状態（よごれ）を読めた**。予報が完璧な世界で、手作りの特徴量のHGBと並ぶ水準（0.210 対 0.214）。
- ただし、**予報が不正確な普通の世界では、系列モデルの利点は出ない**（GRU 0.774 ≒ MLP 0.770）。予報の誤差が支配的。
- **隠れた状態が無い世界では、優位が消える**（対照実験）。強さは本物。
- **時間の構造を使うこと**が重要（窓MLPは大きく悪化）。
- 深層学習の価値は、「**他の誤差が小さく、隠れた状態が重要**」なときに出る。

次の Phase4 では、**データ量とノイズ**を変えて、深層学習が有利になる条件を調べます。実データ並みの短さ（約1.5年）では、どうなるでしょうか。